# PSD analysis

Turns the sealed PSD files of finished ebb-viewer projects (`psd/<run>/psds_final.pkl`) into group spectra, band-power tables and plots, plus workbooks laid out for GraphPad Prism.

**How to use it.** Copy this notebook **and `psd_utils.py`** into your analysis folder (they must sit side by side). Edit the **Settings** cell, then **Run All**. Make a new copy for each analysis.

You get, in `OUT_DIR` (default `psd_analysis/`):

- `figures/` – every plot as a PNG
- `band_power.csv`, `band_power_normalized.csv` – one row per animal x state x channel (x time), one column per band
- `prism_*.xlsx` – spectra and band power, one sheet per channel x state, ready to paste into Prism

**How the numbers are made**

1. Each animal's PSD for each state and channel is read from `psds_final.pkl`. Animals excluded in Sleep QC are left out unless you ask for them.
2. The 0.5-3.5 Hz bins are multiplied by a set of correction factors for the recording hardware's low-frequency roll-off (the "inverse filter").
3. The line-noise peaks are bridged by interpolation: 56-64 Hz (the 60 Hz peak and the gap left by the 60 Hz notch filter, which is on by default) and 118-122 Hz (the 120 Hz peak). If you computed the PSDs without the notch filter, change `INTERP_RANGES` in the Settings cell to 58-62 Hz.
4. A *normalized* spectrum is the interpolated spectrum divided by its own power between 1 and 100 Hz, so it shows the shape of the spectrum independent of overall amplitude.
5. *Band power* is the area under the spectrum in each band (trapezoid rule), for the interpolated and the normalized spectra.

## Settings

In [ ]:
from pathlib import Path

# Sealed PSD files to analyze: {label: path to psds_final.pkl}.
# Use one entry for one time point, or several (e.g. "pre" and "post") to compare time points.
RUNS = {
    "pre": Path("/path/to/your/project/psd/default/psds_final.pkl"),
    # "post": Path("/path/to/your/other/project/psd/default/psds_final.pkl"),
}

# Experimental groups: {"group name": [animal IDs]}. Leave empty for one group with everyone.
GROUPS = {
    # "WT":  ["Mice-1248", "Mice-1253"],
    # "MUT": ["Mice-1251", "Mice-1279"],
}

# Or read the groups from a CSV file instead (this replaces GROUPS). The easiest layout is one
# row per animal, with two columns named "animal" and "group":
#     animal,group
#     Mice-1248,WT
#     Mice-1251,MUT
# One column per group also works: the group name on top, its animal IDs below it:
#     WT,MUT
#     Mice-1248,Mice-1251
#     Mice-1253,Mice-1279
GROUPS_CSV = None                       # e.g. Path("groups.csv")

OUT_DIR = Path("psd_analysis")          # where tables, workbooks and figures go
INCLUDE_EXCLUDED = False                # True keeps animals excluded in Sleep QC
APPLY_INVERSE_FILTER = True             # correct 0.5-3.5 Hz for the hardware roll-off
INTERP_RANGES = [[56, 64], [118, 122]]  # Hz bridged by interpolation to remove line noise.
                                        # LEAVE AS IS if the PSDs were computed with the 60 Hz notch
                                        # filter (the default in ebb-viewer): 56-64 Hz covers the
                                        # notch's 57-63 Hz gap. CHANGE to [[58, 62], [118, 122]] only
                                        # if you turned the notch off. 118-122 Hz is for the 120 Hz
                                        # peak, which the notch does not touch.
NORM_BAND = (1, 100)                    # Hz; spectra are normalized to the power in this band
BANDS = {                               # Hz; band power is computed for each of these
    "1-4": (1, 4), "4-8": (4, 8), "8-12": (8, 12),
    "12-30": (12, 30), "30-70": (30, 70), "70-100": (70, 100),
}
FREQ_RANGE = (0.5, 110)                 # Hz shown on the spectrum plots
COLOR_MAP = None                        # e.g. {"WT": "tab:green", "MUT": "tab:blue"}; None = automatic
SEX_MAP = None                          # e.g. {"Mice-1248": "M", ...}; orders Prism columns males first

## Load

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import psd_utils

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
TIMES = list(RUNS)

df = psd_utils.load_psd_runs(
    RUNS, apply_inverse_filter=APPLY_INVERSE_FILTER, include_excluded=INCLUDE_EXCLUDED
)
if GROUPS_CSV:
    groups = psd_utils.read_groups_csv(GROUPS_CSV)
else:
    groups = GROUPS or {"all": sorted(df["animal"].unique())}
GROUP_ORDER = list(groups)
df = psd_utils.assign_groups(df, groups)
df = psd_utils.interpolate_line_noise(df, ranges=INTERP_RANGES)
print("Line noise interpolated over (Hz):", *INTERP_RANGES)
print("Check this against how your PSDs were computed: 56-64 Hz is for runs that used the 60 Hz notch.")
df = psd_utils.normalize_spectra(df, band=NORM_BAND)

color_map = COLOR_MAP or {g: plt.cm.tab10(i) for i, g in enumerate(GROUP_ORDER)}

# How many animals are in each group (at each time point)
df[["animal", "time", "group"]].drop_duplicates().groupby(["time", "group"]).size().unstack(
    fill_value=0
)

## Spectra by group

Mean spectrum of each group with a standard-error band, for every channel (rows) and state (columns). The number in the legend is the number of animals.

In [ ]:
for time in TIMES:
    part = df[df["time"] == time]
    psd_utils.plot_spectra(
        part, "interp_spectra", color_map=color_map, freq_range=FREQ_RANGE,
        title=f"Spectra ({time})", save_path=FIG_DIR / f"spectra_{time}.png",
    )
    plt.show()
    psd_utils.plot_spectra(
        part, "interp_spectra", color_map=color_map, freq_range=(0.5, 20), log_y=False,
        sharey=False, title=f"Spectra, low frequencies ({time})",
        save_path=FIG_DIR / f"spectra_low_{time}.png",
    )
    plt.show()
    psd_utils.plot_spectra(
        part, "norm_spectra", color_map=color_map, freq_range=(NORM_BAND[0], NORM_BAND[1]),
        title=f"Normalized spectra ({time})", save_path=FIG_DIR / f"spectra_normalized_{time}.png",
    )
    plt.show()

### Comparing time points

Only runs when `RUNS` has more than one entry: each group at each time point, overlaid.

In [ ]:
if len(TIMES) > 1:
    pair_colors = {
        f"{g}_{t}": plt.cm.tab10(i)
        for i, (g, t) in enumerate((g, t) for g in GROUP_ORDER for t in TIMES)
    }
    psd_utils.plot_spectra(
        df, "interp_spectra", group_col="group_time", color_map=pair_colors,
        freq_range=FREQ_RANGE, title="Spectra by group and time",
        save_path=FIG_DIR / "spectra_by_group_and_time.png",
    )
    plt.show()
    psd_utils.plot_spectra(
        df, "norm_spectra", group_col="group_time", color_map=pair_colors,
        freq_range=(NORM_BAND[0], NORM_BAND[1]), title="Normalized spectra by group and time",
        save_path=FIG_DIR / "spectra_normalized_by_group_and_time.png",
    )
    plt.show()

## Spectra by animal

One line per animal. Use these to spot an animal whose spectrum looks different from the rest.

In [ ]:
for time in TIMES:
    for group in GROUP_ORDER:
        part = df[(df["time"] == time) & (df["group"] == group)]
        if part.empty:
            continue
        psd_utils.plot_spectra_by_animal(
            part, "interp_spectra", freq_range=FREQ_RANGE, title=f"{group} by animal ({time})",
            save_path=FIG_DIR / f"by_animal_{group}_{time}.png",
        )
        plt.show()

## Band power

Power in each band, for every animal. In the box plots each dot is an animal; dots outside the usual fences (1.5 x the interquartile range, within each state, channel and group) are outlined and labelled with the animal ID.

In [ ]:
band_raw = psd_utils.band_power_table(df, "interp_spectra", BANDS)
band_norm = psd_utils.band_power_table(df, "norm_spectra", BANDS)
band_raw.to_csv(OUT_DIR / "band_power.csv", index=False)
band_norm.to_csv(OUT_DIR / "band_power_normalized.csv", index=False)

multi = len(TIMES) > 1
box_args = dict(
    x="group", hue="time" if multi else "group", order=GROUP_ORDER,
    hue_order=TIMES if multi else GROUP_ORDER,
    outlier_groups=["state", "ch", "group", "time"],
)
for name, table, ylabel in [
    ("raw", band_raw, r"Power [$\mu V^2$]"),
    ("normalized", band_norm, "Power [a.u.]"),
]:
    for band in BANDS:
        psd_utils.boxplot_band_power(
            table, band, ylabel, f"Band {band} Hz ({name})",
            save_path=FIG_DIR / f"band_{band}_{name}.png", **box_args,
        )
        plt.show()

## Export for Prism

One workbook per time point and kind. Each has one sheet per channel and state; the spectra workbooks have a frequency column and the band-power workbooks a row per band, followed by a block of columns per group (one column per animal), in the order of `GROUPS`.

In [ ]:
for time in TIMES:
    part = df[df["time"] == time]
    for column, name in [("interp_spectra", "spectra"), ("norm_spectra", "spectra_normalized")]:
        psd_utils.export_spectra_prism(
            part, column, GROUP_ORDER, OUT_DIR / f"prism_{name}_{time}.xlsx", sex_map=SEX_MAP
        )
    for table, name in [(band_raw, "band_power"), (band_norm, "band_power_normalized")]:
        psd_utils.export_band_power_prism(
            table[table["time"] == time], list(BANDS), GROUP_ORDER,
            OUT_DIR / f"prism_{name}_{time}.xlsx", sex_map=SEX_MAP,
        )
print("Wrote:", *sorted(p.name for p in OUT_DIR.glob("prism_*.xlsx")), sep="\n  ")